In [5]:
from collections import namedtuple
from time import sleep
import logging

from pulao.logging import init_logging

import pandas as pd

from pulao.trend import TrendManager
from pulao.swing import SwingManager
from pulao.bar import SBar, SBarManager, CBarManager
from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.object import BarData

import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pulao.constant import *

from pulao.logging import logger
init_logging(level=logging.DEBUG)
open("./logs/pulao.log", "w").close() # 每次运行前清空日志
logger.debug("开始运行")

df = pl.read_csv("../dataset/I8888.XDCE_60m.csv", try_parse_dates=True)
df = df.head(100)  # test

sbar_list = []
columns = df.columns

for idx, row in enumerate(df.iter_rows()):
    row_dict = dict(zip(columns, row))
    # datetime,open,close,high,low,volume,money,open_interest,signal
    datetime = row_dict["datetime"]
    o = row_dict["open"]
    close = row_dict["close"]
    high = row_dict["high"]
    low = row_dict["low"]
    volume = row_dict["volume"]
    money = row_dict["money"]
    open_interest = row_dict["open_interest"]

    bar = BarData(
        gateway_name="ctp-test",
        symbol="i8888",
        exchange=Exchange.SHFE,
        interval=Interval.MINUTE,
        datetime=datetime,
        open_price=o,
        close_price=close,
        high_price=high,
        low_price=low,
        volume=volume,
        open_interest=open_interest,
        turnover=money,
    )
    sbar = SBar(symbol=bar.symbol, exchange=bar.exchange.value, interval=bar.interval.value, datetime=datetime, turnover=money, open_price=o, close_price=close, high_price=high, low_price=low, volume=volume)

    sbar_list.append(sbar)
# 模拟行情数据接收
sbar_manager = SBarManager()
cbar_manager = CBarManager(sbar_manager=sbar_manager)
swing_manager = SwingManager(cbar_manager=cbar_manager)

# 流入模拟数据
for sbar in sbar_list:
    sbar_manager.append(sbar)

logger.debug("运行结束")
# sleep(3) # 等一会，让日志输出完
#
# region . Plotly - Cbar
#
df_cbar = cbar_manager.df_cbar.to_pandas()
df_cbar["datetime"] = pd.date_range("2025-01-01", periods=df_cbar.shape[0], freq="h")
df_cbar["open_price"] = df_cbar["high_price"]
df_cbar["close_price"] = df_cbar["low_price"]
df_cbar.insert(0, "index", range(len(df_cbar)))

# region 构建波段数据
df_swing = swing_manager.df_swing.to_pandas()
df_swing["start_datetime"] = pd.Series(dtype="datetime64[ns]")
df_swing["end_datetime"] = pd.Series(dtype="datetime64[ns]")
df_swing.insert(0, "index", range(len(df_swing)))

for i, row in enumerate(df_swing.itertuples()):
    df = df_cbar[(df_cbar["id"] >= row.cbar_start_id) & (df_cbar["id"] <= row.cbar_end_id)]
    start_datetime = df.iloc[0]["datetime"]
    end_datetime = df.iloc[-1]["datetime"]
    df_swing.at[i, "start_datetime"] = start_datetime
    df_swing.at[i, "end_datetime"] = end_datetime

xs = []
ys = []
texts = []
text_positions  = []
for i, row in enumerate(df_swing.itertuples()):
    xs += [row.start_datetime, row.end_datetime, None]
    start_index = df_cbar[(df_cbar["id"] == row.cbar_start_id)]["index"].iloc[0]
    end_index = df_cbar[(df_cbar["id"] == row.cbar_end_id)]["index"].iloc[0]
    if row.direction == Direction.UP:
        start_price = df_cbar[(df_cbar["id"] == row.cbar_start_id)]["low_price"].iloc[0]
        end_price = df_cbar[(df_cbar["id"] == row.cbar_end_id)]["high_price"].iloc[0]
        text_positions += ['bottom center', 'top center', "middle center"]
    else:
        start_price = df_cbar[(df_cbar["id"] == row.cbar_start_id)]["high_price"].iloc[0]
        end_price = df_cbar[(df_cbar["id"] == row.cbar_end_id)]["low_price"].iloc[0]
        text_positions += ['top center', 'bottom center', "middle center"]
    ys += [start_price, end_price, None]
    texts += [start_index, end_index, None]


# endregion

fig = make_subplots(
    rows=1, cols=1
)
fig.add_trace(go.Candlestick(
    x=df_cbar['datetime'],
    open=df_cbar['open_price'],
    high=df_cbar['high_price'],
    low=df_cbar['low_price'],
    close=df_cbar['close_price'],
    name='K线',
), row=1, col=1)

df_fractal_bottom = df_cbar[(df_cbar['fractal_type'] == FractalType.BOTTOM)]

fig.add_trace(go.Scatter(
    x=df_fractal_bottom['datetime'],
    y=df_fractal_bottom['low_price'] * 0.995,   # 放在K线最低价下方一点，不遮挡蜡烛
    mode='markers+text',
    name='swing point low',
    marker=dict(
        symbol='triangle-up',
        size=14,
        color='#1E90FF',
    ),
    text=df_fractal_bottom["index"],      # ← 添加序号
    textposition="bottom center",              # ← 文字位置
    textfont=dict(size=10, color="white"),
    hovertemplate=
        "<b>波段低点</b><br>" +
        "时间: %{x}<br>" +
        "价格: %{y:.2f}<br>" +
        "<extra></extra>"
), row=1, col=1)


df_fractal_top = df_cbar[(df_cbar['fractal_type'] == FractalType.TOP)]

fig.add_trace(go.Scatter(
    x=df_fractal_top['datetime'],
    y=df_fractal_top['high_price'] * 1.005,  # 放在K线最高价上方一点
    mode='markers+text',
    name='swing point high',
    marker=dict(
        symbol='triangle-down',
        size=14,
        color='#FF4500',
    ),
    text=df_fractal_top["index"],      # ← 添加序号
    textposition="top center",              # ← 文字位置
    textfont=dict(size=10, color="white"),
    hovertemplate=
        "<b>波段高点</b><br>" +
        "时间: %{x}<br>" +
        "价格: %{y:.2f}<br>" +
        "<extra></extra>"
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=xs,
    y=ys,
    name='swing segments',
    mode='lines+text',    # lines+text 支持显示文字
    line=dict(width=2, color='orange'),
    text=texts,
    textposition=text_positions,  # 文字位置
))

title = 'Swing Chart - 波段标识'
fig.update_layout(title=title,
                  height=900,
                  hovermode='x unified',  # X轴悬停联动虚线
                  )

fig.update_xaxes(
    showgrid=False,
)

fig.update_yaxes(
    showgrid=False,
)

fig.show()

# endregion



{"event": "开始运行", "level": "debug", "logger": "__main__", "time": "2025-11-28 11:25:02"}
{"cbar_count": 1, "event": "用于组成分形的cbar数量不够", "level": "warning", "logger": "__main__", "time": "2025-11-28 11:25:02"}
{"cbar_count": 1, "event": "用于组成分形的cbar数量不够", "level": "warning", "logger": "__main__", "time": "2025-11-28 11:25:02"}
{"cbar_count": 2, "event": "用于组成分形的cbar数量不够", "level": "warning", "logger": "__main__", "time": "2025-11-28 11:25:02"}
{"fractal": "Fractal(left=CBar(id=120122780241887232, start_id=120122780225110016, end_id=120122780237692928, high_price=942.2349853515625, low_price=935.9240112304688, created_at=datetime.datetime(2025, 11, 28, 11, 25, 2, 583000), fractal_type=0), middle=CBar(id=120122780254470144, start_id=120122780250275840, end_id=120122780250275840, high_price=939.052001953125, low_price=930.2979736328125, created_at=datetime.datetime(2025, 11, 28, 11, 25, 2, 586000), fractal_type=-1), right=CBar(id=120122780267053056, start_id=120122780262858752, end_id=12012

In [4]:
# 测试swing 构建算法
from IPython.display import display
from pulao.logging import logger,init_logging
import logging
from time import sleep
from pulao.swing import Swing,SwingManager
from pulao.bar import CBar,CBarManager,SBarManager
from pulao.constant import EventType
init_logging(level=logging.DEBUG)
with open("./logs/pulao.log", "w"):
    pass # 每次运行前清空日志

logger.debug("开始运行")

def format_df_swing():
    # 在df_swing中显示df_cbar的索引顺序
    import polars as pl
    if "index" in cbar_manager.df_cbar.columns:
        cbar_manager.df_cbar = cbar_manager.df_cbar.drop("index")
    cbar_manager.df_cbar = cbar_manager.df_cbar.with_row_index("index")

    df_cbar = cbar_manager.df_cbar.select(["id","index"])
    df1 = swing_manager.df_swing.join(df_cbar, left_on="start_id", right_on="id", how="left")
    df2 = swing_manager.df_swing.join(df_cbar, left_on="end_id", right_on="id", how="left")
    df1 = df1.rename({"index":"start_index"})
    df2 = df2.rename({"index":"end_index"})
    df2 = df2.select(["id","end_index"])
    df1 = df1.join(df2, on="id", how="left")
    df = df1.select(["id","start_index","end_index","start_id","end_id","high_price","low_price","direction","is_completed","created_at"])

    return df


cbar_manager = CBarManager(sbar_manager=SBarManager())
swing_manager = SwingManager(cbar_manager=cbar_manager)
df_cbar = cbar_manager.read_parquet()
cbar_manager.df_cbar = cbar_manager.df_cbar.slice(0,0)
for i in range(df_cbar.height-1):
    cbar = CBar(**df_cbar.row(i, named=True))
    cbar_manager._append_cbar(start_id=cbar.sbar_start_id, end_id=cbar.sbar_end_id, high_price=cbar.high_price, low_price=cbar.low_price, fractal_type=cbar.fractal_type)
    cbar_manager.notify(EventType.CBAR_CREATED)

logger.debug("运行结束")
sleep(0.2)


display("df_swing")
df = format_df_swing()
display(df)

display("df_cbar")
cbar_manager.df_cbar



{"event": "开始运行", "level": "debug", "logger": "__main__", "time": "2025-11-28 07:58:45"}
{"cbar_count": 1, "event": "用于组成分形的cbar数量不够", "level": "warning", "logger": "__main__", "time": "2025-11-28 07:58:45"}
{"cbar_count": 2, "event": "用于组成分形的cbar数量不够", "level": "warning", "logger": "__main__", "time": "2025-11-28 07:58:45"}
{"fractal": "Fractal(left=CBar(id=120070865390927872, start_id=120070369473200128, end_id=120070369502560256, high_price=942.2349853515625, low_price=935.9240112304688, created_at=datetime.datetime(2025, 11, 28, 7, 58, 45, 118000), fractal_type=0), middle=CBar(id=120070865395122176, start_id=120070369523531776, end_id=120070369523531776, high_price=939.052001953125, low_price=930.2979736328125, created_at=datetime.datetime(2025, 11, 28, 7, 58, 45, 119000), fractal_type=-1), right=CBar(id=120070865395122177, start_id=120070369548697600, end_id=120070369548697600, high_price=951.2730102539062, low_price=937.4929809570312, created_at=datetime.datetime(2025, 11, 28, 7,

'df_swing'

id,start_index,end_index,start_id,end_id,high_price,low_price,direction,is_completed,created_at
u64,u32,u32,u64,u64,f32,f32,i8,bool,datetime[ms]
120070865621614592,1,24,120070865395122176,120070865609031681,988.528015,930.297974,1,true,2025-11-28 07:58:45.173
120070865890050048,24,52,120070865609031681,120070865881661440,988.528015,906.143005,-1,true,2025-11-28 07:58:45.237
120070865978130433,52,67,120070865881661440,120070865973936129,955.924988,906.143005,1,true,2025-11-28 07:58:45.258
120070866154291200,67,98,120070865973936129,120070866137513985,955.924988,842.133972,-1,true,2025-11-28 07:58:45.300
120070866196234240,98,104,120070866137513985,120070866187845632,879.856995,842.133972,1,true,2025-11-28 07:58:45.310
…,…,…,…,…,…,…,…,…,…
120070871317479424,635,641,120070871191650304,120070871271342080,782.034973,752.098022,1,true,2025-11-28 07:58:46.531
120070871439114240,641,653,120070871271342080,120070871426531328,782.034973,739.611023,-1,true,2025-11-28 07:58:46.560
120070871627857920,653,672,120070871426531328,120070871611080704,793.218018,739.611023,1,true,2025-11-28 07:58:46.605


'df_cbar'

index,id,start_id,end_id,high_price,low_price,fractal_type,created_at
u32,u64,u64,u64,f32,f32,i8,datetime[ms]
0,120070865390927872,120070369473200128,120070369502560256,942.234985,935.924011,0,2025-11-28 07:58:45.118
1,120070865395122176,120070369523531776,120070369523531776,939.052002,930.297974,-1,2025-11-28 07:58:45.119
2,120070865395122177,120070369548697600,120070369548697600,951.27301,937.492981,0,2025-11-28 07:58:45.119
3,120070865399316480,120070369620000768,120070369645166592,955.56897,948.888,1,2025-11-28 07:58:45.120
4,120070865403510784,120070369657749504,120070369657749504,950.607971,946.338989,-1,2025-11-28 07:58:45.121
…,…,…,…,…,…,…,…
754,120070872533827584,120070390012706816,120070390012706816,740.593994,730.604004,0,2025-11-28 07:58:46.821
755,120070872538021888,120070390046261248,120070390067232768,741.554016,735.406982,0,2025-11-28 07:58:46.822
756,120070872542216192,120070390084009984,120070390084009984,747.684021,740.830994,0,2025-11-28 07:58:46.823


In [ ]:
# 可视化日志
import pandas as pd
df = pd.read_json("./logs/pulao.log",lines=True)
df = df.drop(columns=["logger","level","time"])
priority_cols = ["event"]
other_cols = [c for c in df.columns if c not in priority_cols]
df = df[priority_cols+other_cols]
df["info"] = df[other_cols].apply(lambda row: row.dropna().tolist(), axis=1)
df = df.drop(columns=other_cols)
df["info"] = df["info"].apply(lambda x: "" if x == [] else x)
df

In [14]:
df = sbar_manager.df_sbar
df = df.with_columns(
    pl.concat_list(["ema_short","ema_long"]).alias("ema")
)
df.lazy().select(pl.col("ema")).explain()

'DF ["id", "datetime", "symbol", "exchange", ...]; PROJECT["ema"] 1/15 COLUMNS'